# 06 · Catalysis with non-covalent contacts

An organocatalysed transition state is a reacting core and the contacts that hold the catalyst to it. This notebook
holds the core of two computed TSs as [05_transition_states](05_transition_states.ipynb) does, and the catalyst's
H-bonds as [04_nci](04_nci.ipynb) does. Each render draws the contacts xyzgraph detects dotted and the partial
bonds dashed.

In [ ]:
import rxembed as rx
import xyzrender
from rdkit import Chem

## A chiral phosphoric acid TS

A BINOL phosphoric acid protonates a dienone as it cyclises: H35 moves from the phosphate O23 to the dienone O75 as a
C-C bond forms, and xyzgraph reads H35...O23 as the H-bond `HB:O75H35->O23`. The core holds the protonated dienone
whole and O23, so that H-bond keeps its TS length of 1.37 angstrom, which an H-bond window (1.6 to 2.2 angstrom)
would not; the rest of the catalyst is embedded anew.

In [ ]:
ts = rx.read_xyz("structures/cpa.xyz", charge=0)
catalyst, substrate = Chem.GetMolFrags(ts)  # xyzgraph puts the moving H on the dienone
MOVING_H, PHOSPHATE_O, DIENONE_O = 35, 23, 75
cpa = rx.embed(ts, fix=[PHOSPHATE_O, *substrate], n=4, seed=1).minimize()
print(cpa)
print(f"H...O {cpa.measure((MOVING_H, PHOSPHATE_O))['mean']:.2f} A")

partial = [(MOVING_H + 1, DIENONE_O + 1), (71, 75)]  # the forming O-H and C...C; xyzrender counts atoms from 1
xyzrender.render(xyzrender.load(cpa.lowest(1).mol, nci_detect=True), ts_bonds=partial)

## A BIMP TS held by its own grips

A bifunctional iminophosphorane (BIMP) catalyst holds an isatin and an aza-allyl anion as two bonds form between them.
`contacts="input"` holds the grips the TS makes: the isatin and the aza-allyl stay where the TS has them, and the
catalyst is embedded anew in its four N-H...O H-bonds to the isatin O's. `rx.contacts(ts, kinds=("HB",))` would
instead enumerate eight grips, each using a sulfonyl O, so for a known TS its observed grips are the ones to hold.

In [ ]:
ts = rx.read_xyz("structures/bimp.xyz", charge=0)
isatin, allyl, catalyst = Chem.GetMolFrags(ts)
bound = rx.embed(ts, fix=[*isatin, *allyl], contacts="input", n=2, seed=1).minimize()
print(bound.tag["nci"])

forming = [(12, 13), (11, 15)]  # O...C and C...C
xyzrender.render(xyzrender.load(bound.lowest(1).mol, nci_detect=True), ts_bonds=forming)